In [1]:
import cv2
import os
import os

print(os.getcwd())
print(os.listdir("models"))


C:\Users\abhil\Age_Access_Control
['.ipynb_checkpoints', 'deploy.prototxt', 'res10_300x300_ssd_iter_140000.caffemodel']


In [19]:
pip install gdown


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
with open("models/deploy.prototxt", "r") as f:
    print(f.readline())


input: "data"



In [5]:
import cv2

face_net = cv2.dnn.readNet(
    "models/res10_300x300_ssd_iter_140000.caffemodel",
    "models/deploy.prototxt"
)

print("✅ Face model loaded successfully")


✅ Face model loaded successfully


In [27]:
import os
import gdown

model_dir = "models"
os.makedirs(model_dir, exist_ok=True)

files = {
    "age_deploy.prototxt": "1jFJc76gRRpQhA2Rr6P3f8L2sbJwisBvL",
    "age_net.caffemodel": "1kWv0AjxGSN0g31OeJa02eBGM0R_jcjIl",
}

for filename, file_id in files.items():
    out_path = os.path.join(model_dir, filename)
    if not os.path.exists(out_path):
        print(f"⬇️ Downloading {filename} from Google Drive...")
        gdown.download(f"https://drive.google.com/uc?export=download&id={file_id}", out_path)
    else:
        print(f"✔️ {filename} already exists.")

print("📦 Model files are ready!")


✔️ age_deploy.prototxt already exists.
✔️ age_net.caffemodel already exists.
📦 Model files are ready!


In [33]:
import cv2

age_net = cv2.dnn.readNet(
    "models/age_net.caffemodel",
    "models/age_deploy.prototxt"
)

print("✅ Age model loaded successfully!")


error: OpenCV(4.12.0) D:\a\opencv-python\opencv-python\opencv\modules\dnn\src\caffe\caffe_io.cpp:1176: error: (-2:Unspecified error) FAILED: ReadProtoFromBinaryFile(param_file, param). Failed to parse NetParameter file: models/age_net.caffemodel in function 'cv::dnn::ReadNetParamsFromBinaryFileOrDie'


In [1]:
AGE_BUCKETS = [
    '(0-2)', '(4-6)', '(8-12)', '(15-20)',
    '(21-32)', '(33-43)', '(44-53)', '(60-100)'
]

ALLOWED_AGES = ['(21-32)', '(33-43)', '(44-53)', '(60-100)']

MODEL_MEAN_VALUES = (
    78.4263377603,
    87.7689143744,
    114.895847746
)


In [3]:
import cv2

# Allowed heuristic instead of age
ALLOWED = True  # Set True if any face grants access

def verify_access_without_age():
    cap = cv2.VideoCapture(0)
    access_granted = False

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        h, w = frame.shape[:2]

        # Convert to blob for face detection (assuming face_net exists)
        blob = cv2.dnn.blobFromImage(frame, 1.0, (300, 300),
                                     (104, 177, 123))
        face_net.setInput(blob)
        detections = face_net.forward()

        for i in range(detections.shape[2]):
            confidence = detections[0, 0, i, 2]

            if confidence > 0.6:
                box = detections[0, 0, i, 3:7] * [w, h, w, h]
                x1, y1, x2, y2 = box.astype(int)

                # Draw rectangle and label
                access_granted = ALLOWED
                label = "ACCESS GRANTED" if access_granted else "ACCESS DENIED"
                color = (0, 255, 0) if access_granted else (0, 0, 255)

                cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
                cv2.putText(frame, label, (x1, y1 - 10),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)

        cv2.imshow("Face Verification (Press Q)", frame)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()
    return access_granted


In [5]:
print("====== AGE BASED APPLICATION LAUNCHER ======")
print("1. Open Browser")
print("2. Open Game (Example)")
print("3. Open Media Player")

choice = input("Enter your choice: ")

if verify_age():
    print("✅ Access granted")

    if choice == "1":
        os.system("start chrome")      # Browser
    elif choice == "2":
        os.system("start notepad")     # Example app/game
    elif choice == "3":
        os.system("start wmplayer")    # Media Player
    else:
        print("Invalid choice")

else:
    print("❌ Access denied")


====== AGE BASED APPLICATION LAUNCHER ======
1. Open Browser
2. Open Game (Example)
3. Open Media Player


Enter your choice:  1


NameError: name 'verify_age' is not defined

In [14]:
import os
import cv2

# Assume face_net is already loaded for face detection
# Example using pre-trained OpenCV DNN face detector
face_net = cv2.dnn.readNet(
    "models/res10_300x300_ssd_iter_140000.caffemodel",
    "models/deploy.prototxt"
)

ALLOWED = True  # Grant access if face is detected

def verify_access():
    cap = cv2.VideoCapture(0)
    access_granted = False

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        h, w = frame.shape[:2]

        blob = cv2.dnn.blobFromImage(frame, 1.0, (300, 300),
                                     (104, 177, 123))
        face_net.setInput(blob)
        detections = face_net.forward()

        for i in range(detections.shape[2]):
            confidence = detections[0, 0, i, 2]
            if confidence > 0.6:
                access_granted = ALLOWED
                x1, y1, x2, y2 = (detections[0, 0, i, 3:7] * [w, h, w, h]).astype(int)
                label = "ACCESS GRANTED" if access_granted else "ACCESS DENIED"
                color = (0, 255, 0) if access_granted else (0, 0, 255)
                cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
                cv2.putText(frame, label, (x1, y1 - 10),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)

        cv2.imshow("Face Verification (Press Q)", frame)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()
    return access_granted


# ================= Launcher =================
print("====== FACE BASED APPLICATION LAUNCHER ======")
print("1. Open Browser")
print("2. Open Game (Example)")
print("3. Open Media Player")

choice = input("Enter your choice: ")

if verify_access():
    print("✅ Access granted")

    if choice == "1":
        os.system("start chrome")      # Browser
    elif choice == "2":
        os.system("start notepad")     # Example app/game
    elif choice == "3":
        os.system("start wmplayer")    # Media Player
    else:
        print("Invalid choice")

else:
    print("❌ Access denied")


====== FACE BASED APPLICATION LAUNCHER ======
1. Open Browser
2. Open Game (Example)
3. Open Media Player


Enter your choice:  1


✅ Access granted
